In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand

# Start Spark session
spark = SparkSession.builder.appName("DistinctExample").getOrCreate()

# Create DataFrame with user_id column of random integers between 0 and 9999
df = spark.range(0, 100000).withColumn("user_id", (rand() * 10000).cast("int"))

# Approximate distinct count using HyperLogLog under the hood
df.selectExpr("approx_count_distinct(user_id)").show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/25 16:58:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/25 16:58:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------+
|approx_count_distinct(user_id)|
+------------------------------+
|                          9823|
+------------------------------+



In [2]:
from pyspark.sql.functions import countDistinct

df.select(countDistinct("user_id")).show()

+-----------------------+
|count(DISTINCT user_id)|
+-----------------------+
|                  10000|
+-----------------------+



In [3]:
spark.stop()

In [18]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand

spark = SparkSession.builder.appName("DistinctExample").getOrCreate()

# Use a different seed each time, based on current time
current_seed = int(time.time())

df = spark.range(0, 100000).withColumn("user_id", (rand(seed=current_seed) * 10000).cast("int"))
df.selectExpr("approx_count_distinct(user_id)").show()

+------------------------------+
|approx_count_distinct(user_id)|
+------------------------------+
|                          9823|
+------------------------------+



In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, rand, window

# Start Spark session
spark = SparkSession.builder.appName("StreamPipeline").getOrCreate()

# Simulated stream: one row per second
stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

# Simulate a 'value' column to work with
data = stream.withColumn("value", (rand(seed=None) * 100).cast("int"))

# Add a synthetic 'user_id' to demonstrate distinct count
data = data.withColumn("user_id", (rand(seed=None) * 1000).cast("int"))

# Define a 10-second tumbling window
aggregated = (
    data.groupBy(window("timestamp", "10 seconds"))
    .agg(
        expr("avg(value) as avg_value"),
        expr("approx_count_distinct(user_id) as est_unique_users")
    )
)

# Output the result to the console
query = (
    aggregated.writeStream
    .outputMode("complete")  # try "update" or "append" as alternatives
    .format("console")
    .option("truncate", False)
    .start()
)

query.awaitTermination()

25/10/25 16:58:40 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/h5/wx50c0ss7pjbwrg0njsbg9lc0000gn/T/temporary-6319535e-deb8-490f-8887-22e46ce863a8. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/10/25 16:58:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+---------+----------------+
|window|avg_value|est_unique_users|
+------+---------+----------------+
+------+---------+----------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+---------+----------------+
|window                                    |avg_value|est_unique_users|
+------------------------------------------+---------+----------------+
|{2025-10-25 16:58:40, 2025-10-25 16:58:50}|43.6     |5               |
+------------------------------------------+---------+----------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-----------------+----------------+
|window                                    |avg_value        |est_unique_users|
+------------------------------------------+-----------------+----------------+
|{2025-10-25 16:58:40, 2025-10-25 16:58:50}|40.53333333333333|15              |
+------------------------------------------+-----------------+----------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+---------+----------------+
|window                                    |avg_value|est_unique_users|
+------------------------------------------+---------+----------------+
|{2025-10-25 16:58:40, 2025-10-25 16:58:50}|41.55    |19              |
+------------------------------------------+---------+----------------+



...

-------------------------------------------
Batch: 35
-------------------------------------------
+------------------------------------------+------------------+----------------+
|window                                    |avg_value         |est_unique_users|
+------------------------------------------+------------------+----------------+
|{2025-10-25 16:59:20, 2025-10-25 16:59:30}|60.111111111111114|9               |
|{2025-10-25 16:58:40, 2025-10-25 16:58:50}|42.58695652173913 |46              |
|{2025-10-25 16:59:10, 2025-10-25 16:59:20}|49.62             |47              |
|{2025-10-25 16:58:50, 2025-10-25 16:59:00}|53.42             |48              |
|{2025-10-25 16:59:00, 2025-10-25 16:59:10}|50.6              |44              |
+------------------------------------------+------------------+----------------+



-------------------------------------------
Batch: 36
-------------------------------------------
+------------------------------------------+------------------+----------------+
|window                                    |avg_value         |est_unique_users|
+------------------------------------------+------------------+----------------+
|{2025-10-25 16:59:20, 2025-10-25 16:59:30}|58.214285714285715|14              |
|{2025-10-25 16:58:40, 2025-10-25 16:58:50}|42.58695652173913 |46              |
|{2025-10-25 16:59:10, 2025-10-25 16:59:20}|49.62             |47              |
|{2025-10-25 16:58:50, 2025-10-25 16:59:00}|53.42             |48              |
|{2025-10-25 16:59:00, 2025-10-25 16:59:10}|50.6              |44              |
+------------------------------------------+------------------+----------------+



ERROR:root:Exception while sending command.                                     
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
RuntimeError: reentrant call inside <_io.BufferedReader name=77>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
        "Error while sending or receiving", e, proto.ERROR_ON_RECEIVE)
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last

-------------------------------------------
Batch: 37
-------------------------------------------
+------------------------------------------+------------------+----------------+
|window                                    |avg_value         |est_unique_users|
+------------------------------------------+------------------+----------------+
|{2025-10-25 16:59:20, 2025-10-25 16:59:30}|54.041666666666664|25              |
|{2025-10-25 16:58:40, 2025-10-25 16:58:50}|42.58695652173913 |46              |
|{2025-10-25 16:59:10, 2025-10-25 16:59:20}|49.62             |47              |
|{2025-10-25 16:58:50, 2025-10-25 16:59:00}|53.42             |48              |
|{2025-10-25 16:59:00, 2025-10-25 16:59:10}|50.6              |44              |
+------------------------------------------+------------------+----------------+



Py4JError: An error occurred while calling o92.awaitTermination

In [5]:
spark.stop()

25/10/25 16:59:31 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 42, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
25/10/25 16:59:31 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 42, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.
25/10/25 16:59:31 ERROR MicroBatchExecution: Query [id = 784049f7-7ea0-4224-98fd-1c629697f6bf, runId = c9f6f970-31f9-423f-94a7-2ba85143528b] terminated with error
org.apache.spark.SparkException: Job 42 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1253)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1251)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1251)
	at org.apache.spark.scheduler.DAGSchedulerEve

25/10/25 16:59:41 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:632)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:610)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:453)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1128)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:628)
	at java.base/java.lang.Thread.